### Gold — Agrégations pour la BI

Construit une table physique prête pour Power BI, croisant ventes et retours par mois/région/catégorie. Choix de conception : table matérialisée plutôt que vue — voir la discussion dans le README du projet pour la justification complète.

**Dépendance : suppose que gold_dimensions.py et gold_facts.py ont déjà tourné.**


In [0]:
from pyspark.sql import functions as F

In [0]:
fact_ventes = spark.table("gold.ventes.fact_ventes")
fact_retours = spark.table("gold.ventes.fact_retours")
dim_temps = spark.table("gold.ventes.dim_temps")
dim_region = spark.table("gold.ventes.dim_region")
dim_produit = spark.table("gold.ventes.dim_produit")

In [0]:
# MAGIC On calcule séparément l'agrégat des ventes et celui des retours, puis on
# MAGIC les joint — plutôt qu'un unique gros JOIN suivi d'un GROUP BY. Pourquoi :
# MAGIC une vente peut avoir zéro, un, ou plusieurs retours ; agréger les deux
# MAGIC faits ensemble avant de grouper risquerait de dupliquer les lignes de
# MAGIC ventes autant de fois qu'il y a de retours associés, et fausserait
# MAGIC totalement le CA total. Séparer puis joindre sur la clé de regroupement
# MAGIC évite ce piège classique ("fan-out" de jointure).

agg_ventes = (
    fact_ventes
    .join(dim_temps.select("sk_temps", "annee", "mois"), on="sk_temps", how="left")
    .join(dim_region, on="sk_region", how="left")
    .join(dim_produit, on="sk_produit", how="left")
    .groupBy("annee", "mois", "region", "categorie")
    .agg(
        F.sum("montant_total").alias("ca_total"),
        F.sum("quantite").alias("quantite_totale"),
        F.countDistinct("id_vente").alias("nb_ventes"),
    )
)

In [0]:
# MAGIC Pour les retours, on doit d'abord remonter jusqu'à `region`/`categorie` en
# MAGIC passant par `fact_ventes` (fact_retours ne connaît que `id_vente`) — c'est
# MAGIC la jointure entre les deux tables de faits mentionnée dans `gold_facts.py`.

agg_retours = (
    fact_retours
    .join(
        fact_ventes.select("id_vente", "sk_region", "sk_produit"),
        on="id_vente",
        how="inner"
    )
    .join(dim_temps.select("sk_temps", "annee", "mois"), on="sk_temps", how="left")
    .join(dim_region, on="sk_region", how="left")
    .join(dim_produit, on="sk_produit", how="left")
    .groupBy("annee", "mois", "region", "categorie")
    .agg(
        F.countDistinct("id_retour").alias("nb_retours"),
        F.countDistinct("id_vente").alias("nb_ventes_avec_retour"),  # <- nouvelle colonne
        F.sum("montant_rembourse").alias("montant_rembourse_total"),
    )
)

In [0]:
kpi_ventes_mensuel = (
    agg_ventes
    .join(agg_retours, on=["annee", "mois", "region", "categorie"], how="left")
    .withColumn("nb_retours", F.coalesce(F.col("nb_retours"), F.lit(0)))
    .withColumn("nb_ventes_avec_retour", F.coalesce(F.col("nb_ventes_avec_retour"), F.lit(0)))
    .withColumn("montant_rembourse_total", F.coalesce(F.col("montant_rembourse_total"), F.lit(0.0)))
    .withColumn(
        "taux_retour_pct",
        F.round(F.col("nb_ventes_avec_retour") / F.col("nb_ventes") * 100, 2)
    )
    .withColumn(
        "retours_par_vente_moy",
        F.round(F.col("nb_retours") / F.col("nb_ventes"), 2)
    )
    .withColumn(
        "ca_net",
        F.round(F.col("ca_total") - F.col("montant_rembourse_total"), 2)
    )
    .orderBy("annee", "mois", "region", "categorie")
)
display(kpi_ventes_mensuel.limit(15))

In [0]:
(
    kpi_ventes_mensuel.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.ventes.kpi_ventes_mensuel")
)

In [0]:
%sql
SELECT *
FROM gold.ventes.kpi_ventes_mensuel
WHERE annee = 2025
ORDER BY taux_retour_pct DESC
LIMIT 10;